In [1]:
import pandas as pd
import os


In [2]:
os.chdir(r'D:\Project Portofolio\Data Analyst, Scientiest, ML, DL, AI\Retail Shrinkage & Loss Prevention Analytics\retail_shrinkage_analytics')

df       = pd.read_csv('data/transactions_with_predictions.csv')
products = pd.read_csv('raw_data/product.csv')


In [3]:
# ── 1. Fact Table: Transactions ───────────────────────────
fact_transactions = df[[
    'BASKET_ID', 'DAY', 'PRODUCT_ID', 'STORE_ID',
    'QUANTITY', 'SALES_VALUE', 'TRANS_TIME', 'WEEK_NO',
    'RETAIL_DISC', 'COUPON_DISC', 'SHRINKAGE_TYPE',
    'IS_FRAUD', 'IF_is_fraud_v3', 'IF_score_v3'
]].copy()

# Konversi TRANS_TIME ke format readable
# 1630 → "16:30"
fact_transactions['TIME_READABLE'] = fact_transactions['TRANS_TIME'].apply(
    lambda x: f"{int(x)//100:02d}:{int(x)%100:02d}" if pd.notna(x) else "00:00"
)

# Tambah kolom hour untuk analisis
fact_transactions['HOUR'] = fact_transactions['TRANS_TIME'].apply(
    lambda x: int(x) // 100 if pd.notna(x) else 0
)

# Tambah kolom risk flag yang readable
fact_transactions['FRAUD_FLAG'] = fact_transactions['IS_FRAUD'].map({
    0: 'Normal', 1: 'Fraud/Shrinkage'
})

print(f"✅ Fact table: {len(fact_transactions):,} rows")
print(fact_transactions.head(3))

✅ Fact table: 201,400 rows
   BASKET_ID  DAY  PRODUCT_ID  STORE_ID  QUANTITY  SALES_VALUE  TRANS_TIME  \
0   99856690  525      988028       370        -3   -27.840719        1300   
1   99901111  624    12427400       370        -5   -32.443743         800   
2   99793981   86    12781326       370        -5   -28.319458         800   

   WEEK_NO  RETAIL_DISC  COUPON_DISC SHRINKAGE_TYPE  IS_FRAUD  IF_is_fraud_v3  \
0       14          0.0          0.0     VOID_ABUSE         1               1   
1       67          0.0          0.0     VOID_ABUSE         1               1   
2       33          0.0          0.0     VOID_ABUSE         1               1   

   IF_score_v3 TIME_READABLE  HOUR       FRAUD_FLAG  
0    -0.548291         13:00    13  Fraud/Shrinkage  
1    -0.570796         08:00     8  Fraud/Shrinkage  
2    -0.566359         08:00     8  Fraud/Shrinkage  


In [4]:
# ── 2. Dimension: Products ────────────────────────────────
dim_products = products[[
    'PRODUCT_ID', 'DEPARTMENT', 
    'COMMODITY_DESC', 'BRAND'
]].drop_duplicates()

print(f"\n✅ Dim products: {len(dim_products):,} rows")


✅ Dim products: 92,353 rows


In [5]:
# ── 3. Dimension: Store Risk ──────────────────────────────
# Hitung store risk dari fact table
dim_stores = fact_transactions.groupby('STORE_ID').agg(
    total_transactions = ('SALES_VALUE', 'count'),
    total_fraud        = ('IS_FRAUD', 'sum'),
    total_loss         = ('SALES_VALUE', lambda x: abs(x[fact_transactions.loc[x.index, 'IS_FRAUD']==1].sum())),
    avg_if_score       = ('IF_score_v3', 'mean')
).reset_index()

dim_stores['fraud_rate_pct'] = (
    dim_stores['total_fraud'] / dim_stores['total_transactions'] * 100
).round(4)

dim_stores['risk_level'] = dim_stores['fraud_rate_pct'].apply(
    lambda x: 'HIGH RISK' if x > 0.2 else ('MEDIUM RISK' if x > 0.1 else 'NORMAL')
)

print(f"\n✅ Dim stores: {len(dim_stores):,} rows")
print(dim_stores['risk_level'].value_counts())


✅ Dim stores: 548 rows
risk_level
HIGH RISK      404
NORMAL         112
MEDIUM RISK     32
Name: count, dtype: int64


In [6]:
# ── Export semua ke Excel ─────────────────────────────────
print("\n⏳ Exporting...")

# Fact table (sample 50k untuk Power BI agar tidak berat)
fact_sample = pd.concat([
    fact_transactions[fact_transactions['IS_FRAUD']==1],  # semua fraud
    fact_transactions[fact_transactions['IS_FRAUD']==0].sample(50000, random_state=42)
])

with pd.ExcelWriter('outputs/powerbi_data.xlsx', engine='openpyxl') as writer:
    fact_sample.to_excel(writer,    sheet_name='fact_transactions', index=False)
    dim_products.to_excel(writer,   sheet_name='dim_products',      index=False)
    dim_stores.to_excel(writer,     sheet_name='dim_stores',        index=False)

print("✅ Exported → outputs/powerbi_data.xlsx")
print(f"\n📋 Summary:")
print(f"   fact_transactions : {len(fact_sample):,} rows")
print(f"   dim_products      : {len(dim_products):,} rows")
print(f"   dim_stores        : {len(dim_stores):,} rows")


⏳ Exporting...
✅ Exported → outputs/powerbi_data.xlsx

📋 Summary:
   fact_transactions : 51,400 rows
   dim_products      : 92,353 rows
   dim_stores        : 548 rows
